# SFT Training: google/gemma-3-4b-it



## 1. Установка зависимостей, импорты, настройка



In [1]:
%%capture
!pip install -q transformers==4.47.1 trl==0.12.2 accelerate==1.2.1 bitsandbytes==0.45.0 peft==0.14.0
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 38.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import os, sys, json, gc, math, shutil, textwrap, warnings
from pathlib import Path
from datetime import datetime
from copy import deepcopy

import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    TrainerCallback,
    BitsAndBytesConfig,
)
from trl import SFTTrainer
from datasets import Dataset

warnings.filterwarnings('ignore')
print(f"PyTorch  : {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.11.0+cpu
Transformers: 5.12.0
CUDA available: False


Мы использовали A100 для дообучения


In [4]:
MODEL_ID = "google/gemma-3-4b-it"

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE   = Path("/content/drive/MyDrive/Colab Notebooks/gemma3_sft")
CKPT_DIR     = DRIVE_BASE / "checkpoints"
DATA_DIR     = Path("/content/drive/MyDrive/Colab Notebooks")
FINAL_DIR    = DRIVE_BASE / "final_model"

for d in [CKPT_DIR, DATA_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Директории созданы:")
print(f"  Чекпоинты : {CKPT_DIR}")
print(f"  Финальная : {FINAL_DIR}")

Mounted at /content/drive
Директории созданы:
  Чекпоинты : /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints
  Финальная : /content/drive/MyDrive/Colab Notebooks/gemma3_sft/final_model


## 2. Загрузка и валидация данных


In [5]:
#Валидация JSON-файлов
def validate_and_load(path: Path, name: str) -> list:
    """Загружает JSON, проверяет формат, фильтрует пустые примеры."""
    assert path.exists(), f"Файл не найден: {path}"
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    assert isinstance(data, list), f"{name}: ожидался list, получен {type(data)}"

    clean, skipped = [], 0
    for idx, item in enumerate(data):
        assert isinstance(item, dict), f"{name}[{idx}]: элемент не dict"
        assert 'conversations' in item, f"{name}[{idx}]: нет поля 'conversations'"
        convs = item['conversations']
        assert isinstance(convs, list) and len(convs) >= 2, \
            f"{name}[{idx}]: 'conversations' должен быть списком ≥ 2 сообщений"

        # Проверяем роли и непустые значения
        valid = True
        for msg in convs:
            assert 'from' in msg and 'value' in msg, \
                f"{name}[{idx}]: сообщение без 'from' или 'value'"
            if not msg['value'].strip():
                valid = False  # пропускаем примеры с пустым ответом
                skipped += 1
                break

        if valid:
            clean.append(item)

    print(f"  [{name}] Загружено: {len(data)}, отфильтровано пустых: {skipped}, итого: {len(clean)}")
    return clean


raw = {
    'easy'         : validate_and_load(DATA_DIR / 'easy.json',         'easy'),
    'medium_part1' : validate_and_load(DATA_DIR / 'medium_part1.json', 'medium_part1'),
    'medium_part2' : validate_and_load(DATA_DIR / 'medium_part2.json', 'medium_part2'),
    'hard'         : validate_and_load(DATA_DIR / 'hard.json',         'hard'),
}

  [easy] Загружено: 8077, отфильтровано пустых: 0, итого: 8077
  [medium_part1] Загружено: 12154, отфильтровано пустых: 1, итого: 12153
  [medium_part2] Загружено: 12155, отфильтровано пустых: 0, итого: 12155
  [hard] Загружено: 6479, отфильтровано пустых: 0, итого: 6479


## 3. Загрузка токенизатора и модели

In [6]:
# HuggingFace токен
from huggingface_hub import login
login()

In [7]:
# Гиперпараметры
MODEL_MAX_LENGTH          = 1024
LEARNING_RATE             = 2e-5
WEIGHT_DECAY              = 0.0
WARMUP_RATIO              = 0.03
LR_SCHEDULER_TYPE         = "cosine"
GRADIENT_ACCUMULATION     = 16
LOGGING_STEPS             = 1
PER_DEVICE_TRAIN_BATCH    = 1
DTYPE                     = torch.bfloat16

In [19]:
print(f"Загружаем токенизатор: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    model_max_length=MODEL_MAX_LENGTH,
    padding_side='right',
)

# Gemma уже имеет pad_token, но проверим
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("pad_token установлен в eos_token")

print(f"Vocab size : {len(tokenizer)}")
print(f"pad_token  : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"eos_token  : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"bos_token  : {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")

Загружаем токенизатор: google/gemma-3-4b-it


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Vocab size : 262145
pad_token  : '<pad>' (id=0)
eos_token  : '<eos>' (id=1)
bos_token  : '<bos>' (id=2)


In [ ]:
def load_base_model(checkpoint_path: str = None) -> AutoModelForCausalLM:
    """
    Загружает модель из HuggingFace Hub
    """
    src = checkpoint_path if checkpoint_path else MODEL_ID
    print(f"Загружаем модель из: {src}")

    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=DTYPE,
        device_map="auto",
        attn_implementation="eager",
    )
    model.config.use_cache = False
    model.enable_input_require_grads()  # нужно для gradient checkpointing + SFT
    print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
    return model


# Загружаем начальную модель
model = load_base_model()
print("Модель загружена")

Загружаем модель из: google/gemma-3-4b-it


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Параметров: 4.30B
Модель загружена


## 4. Вспомогательные функции

In [ ]:
#Конвертация данных в chat-формат
def conversations_to_text(sample: dict) -> dict:
    """
    Конвертирует элемент датасета в строку через apply_chat_template.
    Формат: [{role: user, content: ...}, {role: assistant, content: ...}, ...]
    """
    messages = []
    role_map = {'human': 'user', 'gpt': 'assistant', 'system': 'system'}
    for msg in sample['conversations']:
        role = role_map.get(msg['from'], msg['from'])
        messages.append({'role': role, 'content': msg['value']})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


def make_dataset(data: list) -> Dataset:
    """Создаёт HF Dataset из списка conversation-примеров."""
    dataset = Dataset.from_list(data)
    dataset = dataset.map(conversations_to_text, num_proc=4, desc='Форматирование')
    return dataset


# Проверка форматирования
sample_text = conversations_to_text(raw['easy'][0])['text']
print("Пример форматирования:")
print(sample_text[:500])
print(f"\nДлина: {len(tokenizer(sample_text)['input_ids'])} токенов")

Пример форматирования:
<bos><start_of_turn>user
Верно ли, что биология — это наука о живых организмах и их взаимодействии с окружающей средой?<end_of_turn>
<start_of_turn>model
верно<end_of_turn>


Длина: 38 токенов


In [11]:
#Sanity check: генерация текста
#Мы включили этот этап, потому что при предыдущей попытке дообучения у нас модель где-то сломалась
#В итоге она генерировала только паддинг, поэтому теперь мы будем проверять это сразу
#Если модель где-то сломается, то мы начнем с последнего чекпоинта
SANITY_PROMPTS = [
    "Что такое фотосинтез?",
    "Назовите три органоида клетки и их функции.",
    "Почему вода является универсальным растворителем?",
]

@torch.inference_mode()
def run_sanity_check(model, tokenizer, epoch_label: str, max_new_tokens: int = 300):
    """
    Проверяет, что модель генерирует осмысленный текст после эпохи.
    Детектирует зависание на pad-токенах.
    Возвращает True если модель в норме.
    """
    print(f"\n{'='*60}")
    print(f"SANITY CHECK после {epoch_label}")
    print(f"{'='*60}")

    model.eval()
    model.config.use_cache = True
    ok = True

    for prompt in SANITY_PROMPTS:
        messages = [{'role': 'user', 'content': prompt}]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
        input_len = inputs['input_ids'].shape[1]

        try:
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        except Exception as e:
            print(f"❌ Ошибка генерации: {e}")
            ok = False
            continue

        generated_ids = output[0][input_len:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        #Диагностика паддинга
        pad_id  = tokenizer.pad_token_id
        pad_count = (generated_ids == pad_id).sum().item()
        total_gen = len(generated_ids)
        pad_ratio  = pad_count / max(total_gen, 1)

        # Считаем уникальные токены (разнообразие)
        unique_tokens = len(set(generated_ids.tolist()))

        print(f"\nPrompt : {prompt}")
        print(f"Answer : {generated_text[:300]}")
        print(f"Stats  : {total_gen} токенов, {unique_tokens} уникальных, pad доля={pad_ratio:.2%}")

        # Критерии сломанной модели
        if pad_ratio > 0.5:
            print(f"⚠️ >50% токенов pad!")
            ok = False
        if unique_tokens <= 3 and total_gen > 10:
            print(f"⚠️ Мало уникальных токенов ({unique_tokens})!")
            ok = False
        if not generated_text.strip():
            print(f"⚠️ Пустая генерация!")
            ok = False

    if ok:
        print(f"\n✅ Санити-чек пройден после {epoch_label}")
    else:
        print(f"\n❌ ПРОБЛЕМА ОБНАРУЖЕНА после {epoch_label}")

    model.train()
    model.config.use_cache = False
    return ok

In [10]:
# Callback: сбор лоссов
class LossLoggerCallback(TrainerCallback):
    """Собирает train loss по шагам для последующего анализа."""
    def __init__(self):
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.losses.append({
                'step'  : state.global_step,
                'loss'  : logs['loss'],
                'epoch' : state.epoch,
                'lr'    : logs.get('learning_rate', None),
            })

    def summary(self, epoch_label: str):
        if not self.losses:
            print("  Нет данных о лоссе.")
            return
        losses = [x['loss'] for x in self.losses]
        print(f"  Лосс [{epoch_label}]:")
        print(f"    Первый шаг : {losses[0]:.4f}")
        print(f"    Последний  : {losses[-1]:.4f}")
        print(f"    Минимум    : {min(losses):.4f}")
        print(f"    Максимум   : {max(losses):.4f}")
        print(f"    Среднее    : {sum(losses)/len(losses):.4f}")
        self.losses.clear()  # сбрасываем для следующей эпохи


#Сохранение чекпоинта
#Дообучение идет примерно 7 часов + после какой-то эпохи может сломаться
#Надо перестраховаться
def save_checkpoint(model, tokenizer, epoch_label: str):
    ckpt_path = CKPT_DIR / epoch_label
    ckpt_path.mkdir(parents=True, exist_ok=True)
    print(f"  Сохраняем чекпоинт: {ckpt_path}")
    model.save_pretrained(str(ckpt_path))
    tokenizer.save_pretrained(str(ckpt_path))

    meta = {
        'epoch_label': epoch_label,
        'timestamp'  : datetime.now().isoformat(),
        'model_id'   : MODEL_ID,
    }
    with open(ckpt_path / 'training_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Чекпоинт сохранён: {ckpt_path}")
    return str(ckpt_path)

## 5. Основная функция обучения одной эпохи

In [12]:
def train_one_epoch(
    model,
    tokenizer,
    dataset: Dataset,
    epoch_label: str,
    output_dir: str,
    loss_logger: LossLoggerCallback,
    num_train_epochs: int = 1,
) -> tuple:
    """
    Обучает модель на датасете в течение num_train_epochs эпох.
    Возвращает (trainer, metrics).
    """
    model.train()
    model.config.use_cache = False

    training_args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = num_train_epochs,
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION,
        learning_rate               = LEARNING_RATE,
        weight_decay                = WEIGHT_DECAY,
        warmup_ratio                = WARMUP_RATIO,
        lr_scheduler_type           = LR_SCHEDULER_TYPE,
        logging_steps               = LOGGING_STEPS,
        save_strategy               = "no",
        eval_strategy               = "no",
        bf16                        = (DTYPE == torch.bfloat16),
        fp16                        = False,
        gradient_checkpointing      = True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers      = 2,
        remove_unused_columns       = False,
        optim                       = "adamw_torch_fused",
        report_to                   = "none",
        logging_first_step          = True,
        seed                        = 42,
    )


    trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    processing_class = tokenizer,
    train_dataset    = dataset,
    )

    print(f"\n{'#'*60}")
    print(f"# Начинаем обучение: {epoch_label}")
    print(f"# Примеров: {len(dataset)}, Эпох: {num_train_epochs}")
    total_steps = math.ceil(len(dataset) / (PER_DEVICE_TRAIN_BATCH * GRADIENT_ACCUMULATION)) * num_train_epochs
    print(f"# Шагов оптимизатора (приблизительно): {total_steps}")
    print(f"{'#'*60}")

    metrics = trainer.train()
    print(f"\nОбучение завершено. Loss: {metrics.training_loss:.4f}")
    return trainer, metrics

## 6. Полный цикл обучения

Сначала отформатируем и проверим данные для дообучения.
Потом уже собственно дообучение.
После каждой эпохи:
1. Лог лоссов
2. Санити-чек
3. Чекпоинт на Диск


In [13]:
# Расписание обучения
SCHEDULE = [
    ('easy',         2),
    ('medium_part1', 1),
    ('medium_part2', 1),
    ('hard',         2),
]

print("Расписание обучения:")
total_ep = 0
for ds_name, n_ep in SCHEDULE:
    total_ep += n_ep
    print(f"  {ds_name:16s} → {n_ep} эпох(а)")
print(f"  Всего эпох: {total_ep}")

Расписание обучения:
  easy             → 2 эпох(а)
  medium_part1     → 1 эпох(а)
  medium_part2     → 1 эпох(а)
  hard             → 2 эпох(а)
  Всего эпох: 6


In [20]:
#Предварительно форматируем все датасеты
datasets_formatted = {}
for ds_name, _ in SCHEDULE:
    if ds_name not in datasets_formatted:
        print(f"  {ds_name}...")
        datasets_formatted[ds_name] = make_dataset(raw[ds_name])
        print(f"    {len(datasets_formatted[ds_name])} примеров готовы")
print("\nВсе датасеты готовы")

  easy...


Форматирование (num_proc=4):   0%|          | 0/8077 [00:00<?, ? examples/s]

    8077 примеров готовы
  medium_part1...


Форматирование (num_proc=4):   0%|          | 0/12153 [00:00<?, ? examples/s]

    12153 примеров готовы
  medium_part2...


Форматирование (num_proc=4):   0%|          | 0/12155 [00:00<?, ? examples/s]

    12155 примеров готовы
  hard...


Форматирование (num_proc=4):   0%|          | 0/6479 [00:00<?, ? examples/s]

    6479 примеров готовы

Все датасеты готовы


In [ ]:
def convert_to_chatml(example):
    role_map = {"human": "user", "gpt": "assistant"}
    messages = [
        {
            "role": role_map.get(msg["from"], msg["from"]),
            "content": msg["value"]
        }
        for msg in example["conversations"]
    ]
    return {"messages": messages}

datasets_formatted = {
    name: ds.map(convert_to_chatml, remove_columns=ds.column_names)
    for name, ds in datasets_formatted.items()
}

Map:   0%|          | 0/8077 [00:00<?, ? examples/s]

Map:   0%|          | 0/12153 [00:00<?, ? examples/s]

Map:   0%|          | 0/12155 [00:00<?, ? examples/s]

Map:   0%|          | 0/6479 [00:00<?, ? examples/s]

In [ ]:
for name, ds in datasets_formatted.items():
    print(f"{name}: {len(ds)} примеров")
    print(f"  Колонки: {ds.column_names}")
    print(f"  Первый пример: {ds[0]['messages']}")

easy: 8077 примеров
  Колонки: ['messages']
  Первый пример: [{'role': 'user', 'content': 'Верно ли, что биология — это наука о живых организмах и их взаимодействии с окружающей средой?'}, {'role': 'assistant', 'content': 'верно'}]
medium_part1: 12153 примеров
  Колонки: ['messages']
  Первый пример: [{'role': 'user', 'content': 'Сформулируйте определение биологии, используя термины «изучение» и «организмы».'}, {'role': 'assistant', 'content': 'Биология — это наука, изучающая организмы и их взаимодействие с окружающей средой.'}]
medium_part2: 12155 примеров
  Колонки: ['messages']
  Первый пример: [{'role': 'user', 'content': 'Что такое географическая среда и какие компоненты она включает?'}, {'role': 'assistant', 'content': 'Географическая среда — это совокупность природных (например, климат, рельеф, водные ресурсы) и антропогенных (например, города, дороги, сельскохозяйственные угодья) компонентов, которые окружают человека и с которыми он взаимодействует.'}]
hard: 6479 примеров
  Ко

In [ ]:
def is_valid_conversation(example):
    messages = example.get("messages", [])

    # Минимум 2 сообщения
    if len(messages) < 2:
        return False

    # Первое должно быть от user
    if messages[0]["role"] != "user":
        return False

    # Роли должны чередоваться
    for i, msg in enumerate(messages):
        expected_role = "user" if i % 2 == 0 else "assistant"
        if msg["role"] != expected_role:
            return False

        # Нет пустого контента
        if not msg.get("content", "").strip():
            return False

    return True


datasets_formatted = {
    name: ds.filter(is_valid_conversation)
    for name, ds in datasets_formatted.items()
}

Filter:   0%|          | 0/8077 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12153 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12155 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6479 [00:00<?, ? examples/s]

In [ ]:
for name, ds in datasets_formatted.items():
    total   = len(ds)
    valid   = ds.filter(is_valid_conversation)
    invalid = total - len(valid)
    print(f"{name}: всего {total}, невалидных {invalid} ({100*invalid/total:.1f}%)")

Filter:   0%|          | 0/8077 [00:00<?, ? examples/s]

easy: всего 8077, невалидных 0 (0.0%)


Filter:   0%|          | 0/12153 [00:00<?, ? examples/s]

medium_part1: всего 12153, невалидных 0 (0.0%)


Filter:   0%|          | 0/12155 [00:00<?, ? examples/s]

medium_part2: всего 12155, невалидных 0 (0.0%)


Filter:   0%|          | 0/6479 [00:00<?, ? examples/s]

hard: всего 6479, невалидных 0 (0.0%)


In [ ]:
for name, ds in datasets_formatted.items():
    print(f"{name}: {len(ds)} примеров")
    if len(ds) > 0:
        print(f"  Колонки: {ds.column_names}")
        print(f"  Первый пример: {ds[0]}")

easy: 8077 примеров
  Колонки: ['id', 'conversations', 'text']
  Первый пример: {'id': '5_класс_easy_0', 'conversations': [{'from': 'human', 'value': 'Верно ли, что биология — это наука о живых организмах и их взаимодействии с окружающей средой?'}, {'from': 'gpt', 'value': 'верно'}], 'text': '<bos><start_of_turn>user\nВерно ли, что биология — это наука о живых организмах и их взаимодействии с окружающей средой?<end_of_turn>\n<start_of_turn>model\nверно<end_of_turn>\n'}
medium_part1: 12153 примеров
  Колонки: ['id', 'conversations', 'text']
  Первый пример: {'id': '5_класс_medium_306', 'conversations': [{'from': 'human', 'value': 'Сформулируйте определение биологии, используя термины «изучение» и «организмы».'}, {'from': 'gpt', 'value': 'Биология — это наука, изучающая организмы и их взаимодействие с окружающей средой.'}], 'text': '<bos><start_of_turn>user\nСформулируйте определение биологии, используя термины «изучение» и «организмы».<end_of_turn>\n<start_of_turn>model\nБиология — это 

In [ ]:
# ЦИКЛ ОБУЧЕНИЯ
loss_logger   = LossLoggerCallback()
epoch_counter = 0
all_checkpoints = []
sanity_results  = {}

for ds_name, n_epochs in SCHEDULE:
    dataset = datasets_formatted[ds_name]

    for ep in range(1, n_epochs + 1):
        epoch_counter += 1
        epoch_label = f"epoch{epoch_counter:02d}_{ds_name}_ep{ep}of{n_epochs}"
        tmp_output  = f"/content/tmp_trainer_{epoch_label}"

        # Обучение
        trainer, metrics = train_one_epoch(
            model        = model,
            tokenizer    = tokenizer,
            dataset      = dataset,
            epoch_label  = epoch_label,
            output_dir   = tmp_output,
            loss_logger  = loss_logger,
            num_train_epochs = 1,
        )

        # Лог лоссов
        print(f"\nЛосс эпохи {epoch_label}:")
        loss_logger.summary(epoch_label)

        # Санити-чек
        sanity_ok = run_sanity_check(model, tokenizer, epoch_label)
        sanity_results[epoch_label] = sanity_ok

        # Чекпоинт
        ckpt_path = save_checkpoint(model, tokenizer, epoch_label)
        all_checkpoints.append((epoch_label, ckpt_path, sanity_ok))

        # Чистим временную директорию трейнера
        if Path(tmp_output).exists():
            shutil.rmtree(tmp_output, ignore_errors=True)

        # Освобождаем память
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

        if not sanity_ok:
            print(f"\n Модель повреждена после {epoch_label}!")
            print(f" Последний рабочий чекпоинт: {all_checkpoints[-2][1] if len(all_checkpoints) > 1 else 'нет'}")
            # Не прерываем, но предупреждение будет видно


print("\n" + "="*60)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*60)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/8077 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.



############################################################
# Начинаем обучение: epoch01_easy_ep1of2
# Примеров: 8077, Эпох: 1
# Шагов оптимизатора (приблизительно): 505
############################################################


Step,Training Loss
1,7.242979
2,7.062360
3,6.923800
4,5.908735
5,6.022529
6,4.313384
7,4.225109
8,3.600398
9,2.924957
10,2.640937


...


############################################################
# Начинаем обучение: epoch06_hard_ep2of2
# Примеров: 6479, Эпох: 1
# Шагов оптимизатора (приблизительно): 405
############################################################


Step,Training Loss
1,0.547906
2,0.647671
3,0.537056
4,0.538794
5,0.499216
6,0.571382
7,0.473151
8,0.459552
9,0.493168
10,0.398554



Обучение завершено. Loss: 0.7150

Лосс эпохи epoch06_hard_ep2of2:
  Нет данных о лоссе.

SANITY CHECK после epoch06_hard_ep2of2

Prompt : Что такое фотосинтез?
Answer : Фотосинтез — это процесс преобразования световой энергии в химическую, осуществляемый растениями, водорослями и некоторыми бактериями для синтеза органических веществ из углекислого газа и воды.
model
Фотосинтез — это процесс преобразования световой энергии в химическую, осуществляемый растениями, в
Stats  : 150 токенов, 44 уникальных, pad доля=0.00%

Prompt : Назовите три органоида клетки и их функции.
Answer : Ядро — хранение и передача наследственной информации, митохондрии — производство энергии в виде АТФ, лизосомы — расщепление биополимеров.
model
Ядро, митохондрии, лизосомы
model
Ядро — хранение и передача наследственной информации, митохондрии — производство энергии в виде АТФ, лизосомы — расщеплени
Stats  : 150 токенов, 40 уникальных, pad доля=0.00%

Prompt : Почему вода является универсальным растворителем?
A

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ Чекпоинт сохранён: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2

ОБУЧЕНИЕ ЗАВЕРШЕНО
